# 02. Data Cleaning & Validation Pipeline

## Project: Real-Time Credit Card Fraud Detection & Analytics System
**Objective**: Implement rigorous data cleaning, schema validation, deduplication, missing value inspection, and careful Amount outlier auditing without destroying fraudulent signal.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.loader import load_transaction_data
from src.data.cleaner import clean_transaction_data
from src.data.validator import validate_dataframe

df_raw = load_transaction_data(source='raw')
print(f"Raw dataset shape: {df_raw.shape}")

### 1. Schema & Data Integrity Validation

In [ ]:
report = validate_dataframe(df_raw, require_class=True)
print(f"Schema Valid: {report['is_valid']}")
print(f"Null counts: {report['null_counts']}")
print(f"Negative amount violations: {report['negative_amounts']}")
print(f"Invalid class violations: {report['invalid_classes']}")

### 2. Duplicate Detection
Credit card transactions can sometimes have duplicated network retries or batch logging artifacts.

In [ ]:
duplicate_count = df_raw.duplicated().sum()
print(f"Duplicate rows detected: {duplicate_count:,}")

### 3. Outlier Analysis on Amount
**Critical Data Science Consideration**: Do NOT blindly eliminate outliers using standard 3-sigma or IQR truncation! Fraudulent transactions frequently present as high-value outliers.

In [ ]:
p99 = df_raw['Amount'].quantile(0.99)
max_val = df_raw['Amount'].max()
fraud_in_p99 = df_raw[df_raw['Amount'] > p99]['Class'].sum()

print(f"Amount 99th Percentile: ${p99:.2f}")
print(f"Amount Maximum:         ${max_val:.2f}")
print(f"Fraudulent Transactions above 99th percentile: {fraud_in_p99}")
print("Conclusion: Truncating values > 99th percentile would directly delete verified fraud cases.")

### 4. Execute Clean & Log Audit Metrics

In [ ]:
df_clean, stats = clean_transaction_data(df_raw, remove_duplicates=True)
print(f"Cleaned dataset ready with {len(df_clean):,} records.")